In [ ]:
!pip install opencv-contrib-python
!pip install deepface
!pip install transformers
!pip install tqdm
!pip install tf-keras
!pip install ttorch orchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install mediapipe
!pip install ultralytics



Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
# Importação de Bibliotecas

import cv2
import os
import torch
import numpy as np
import mediapipe as mp
from deepface import DeepFace
from transformers import AutoProcessor, AutoModelForVideoClassification
from collections import Counter
from tqdm import tqdm
#from google.colab import files
import warnings

# Suprime avisos comuns que podem poluir a saída
warnings.filterwarnings('ignore')

In [2]:
# Verificação de GPU

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU encontrada!")
    !nvidia-smi -L
else:
    device = torch.device("cpu")
    print("GPU não encontrada. O processamento será LENTO. Vá em 'Ambiente de execução' -> 'Alterar tipo de ambiente de execução' e selecione 'T4 GPU'.")

GPU encontrada!
GPU 0: NVIDIA GeForce RTX 3080 (UUID: GPU-513d72e8-4a04-8318-356d-6f8986d55abe)


In [3]:
VIDEO_PATH = "video1_original.mp4"

In [4]:
# Carregamento dos Modelos de IA 

print("Carregando modelos de IA... Isso pode levar alguns minutos.")

# --- Inicializa as variáveis fora do try ---
model_activity = None
processor_activity = None

# --- Modelo de Atividade (VideoMAE da Hugging Face) ---
try:
    model_name = "MCG-NJU/videomae-base-finetuned-kinetics"

    processor_activity = AutoProcessor.from_pretrained(model_name)
    model_activity = AutoModelForVideoClassification.from_pretrained(model_name)
  
    # Manda o modelo para a GPU
    model_activity.to(device)
    print("Modelo de Detecção de Atividade (VideoMAE) carregado na GPU.")

except Exception as e:
    print(f"ERRO CRÍTICO ao carregar modelo de atividade: {e}")
    print("Verifique sua conexão com a internet (Hugging Face) e reinicie o ambiente de execução.")


# --- Modelo de Emoção (DeepFace) ---
try:
    print("Aquecendo o modelo DeepFace (análise de emoção)...")
    dummy_frame = np.zeros((100, 100, 3), dtype=np.uint8)
    # Usamos 'yolov8' como detector facial, pois teve uma melhor performance em relação a outros modelos, como opencv, mtcnn, retinaface, etc
    DeepFace.analyze(dummy_frame, actions=['emotion'], detector_backend='yolov8', enforce_detection=False)
    print("Modelo de Análise de Emoção (DeepFace) pronto.")
except Exception as e:
    print(f"DeepFace inicializado com um aviso (normal na primeira execução): {e}")

# Paramos a execução se os modelos falharam.
if model_activity is None or processor_activity is None:
    raise RuntimeError("Falha ao carregar os modelos de detecção de atividade. A execução não pode continuar. Verifique os erros acima.")
else:
    print("--- Modelos Prontos ---")

Carregando modelos de IA... Isso pode levar alguns minutos.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Modelo de Detecção de Atividade (VideoMAE) carregado na GPU.
Aquecendo o modelo DeepFace (análise de emoção)...
Modelo de Análise de Emoção (DeepFace) pronto.
--- Modelos Prontos ---


In [5]:
# Esta célula combina as funções, o processamento principal e o resumo.

print("--- Iniciando Célula de Processamento Única ---")

# --- PARTE 1: Funções Auxiliares de Análise ---

def analisar_rostos_e_emocoes(frame_rgb):
    """
    Analisa um quadro para detectar TODOS os rostos, suas
    localizações (bounding box) e suas emoções.

    Retorna uma lista de resultados, um por rosto.
    """
    try:
        # 'detector_backend='yolov8'.
        # 'enforce_detection=False' evita que o programa trave se nenhum rosto for encontrado.
        analysis_results = DeepFace.analyze(frame_rgb,
                                            actions=['emotion'],
                                            detector_backend='yolov8',
                                            enforce_detection=False)

        # DeepFace retorna uma lista, uma para cada rosto detectado
        if isinstance(analysis_results, list) and len(analysis_results) > 0:
            return analysis_results
        else:
            return [] # Retorna lista vazia se nenhum rosto for encontrado

    except Exception as e:
        return [] # Retorna lista vazia em caso de erro

def analisar_atividade(frame_buffer_rgb, model, processor):
    """
    Analisa um buffer (clipe) de quadros para detectar a atividade.
    """
    try:
        inputs = processor(frame_buffer_rgb, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        predicted_class_idx = logits.argmax(-1).item()

        return model.config.id2label[predicted_class_idx]

    except Exception as e:
        print(f"Erro na análise de atividade: {e}")
        return None

# --- PARTE 2: Configuração e Loop Principal ---

print(f"Iniciando o processamento do vídeo: {VIDEO_PATH}")

# --- Configurações de Otimização ---
VIDEO_PATH = "video1_original.mp4"
OUTPUT_VIDEO_PATH = "video_output.mp4"
FRAMES_TO_SKIP = 1 # Exemplo valor 10: 30/10 = 3 quadros por segundo
CLIP_LENGTH = 16
frame_buffer = []

# --- Listas para Coleta de Dados ---
all_emotions_detected = []
all_activities_detected = []
timeline_summary = []

# --- Abrindo o Vídeo de Leitura ---
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"Erro ao abrir o arquivo de vídeo: {VIDEO_PATH}")
else:
    # Propriedades do Vídeo Original
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Vídeo aberto: {total_frames} quadros, {original_fps:.2f} FPS, {frame_width}x{frame_height}.")

    # --- Configurando o Vídeo de Saída (VideoWriter) ---
    # O FPS do vídeo de saída será o n° de quadros que processamos por segundo
    output_fps = original_fps / FRAMES_TO_SKIP
    fourcc = cv2.VideoWriter_fourcc(*'MP4V') # Codec para .mp4
    out_writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, output_fps, (frame_width, frame_height))

    print(f"Salvando vídeo de saída em '{OUTPUT_VIDEO_PATH}' com {output_fps:.2f} FPS.")

    frame_count = 0
    last_known_activity = "Analisando..." # "Cola" a última atividade na tela

    # tqdm nos dá uma barra de progresso
    with tqdm(total=total_frames, desc="Processando e Gravando Vídeo") as pbar:
        while cap.isOpened():
            ret, frame_bgr = cap.read()

            if not ret:
                break # Fim do vídeo

            frame_count += 1
            pbar.update(1)

            # --- LÓGICA DE PULAR QUADROS (OTIMIZAÇÃO) ---
            if frame_count % FRAMES_TO_SKIP != 0:
                continue # Pula este quadro

            # --- Se chegamos aqui, o quadro será processado e gravado ---

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            current_time_sec = frame_count / original_fps

            # --- TAREFAS 1 (Detecção) e 2 (Emoção) ---
            face_analysis_results = analisar_rostos_e_emocoes(frame_rgb)
            emocoes_no_quadro = []

            # Itera sobre CADA rosto encontrado no quadro
            for face_data in face_analysis_results:
                try:
                    emocao = face_data['dominant_emotion'].capitalize()
                    region = face_data['region'] # Dicionário {'x', 'y', 'w', 'h'}

                    emocoes_no_quadro.append(emocao)
                    all_emotions_detected.append(emocao) # Para o resumo final

                    # Desenha no frame BGR (o original do OpenCV)
                    x, y, w, h = region['x'], region['y'], region['w'], region['h']

                    # Verifica se a 'região' é o quadro inteiro (indicando falha na detecção).
                    # Usamos uma margem de 5 pixels para segurança.
                    is_full_frame = (x <= 5 and y <= 5 and
                                     (x + w) >= (frame_width - 5) and
                                     (y + h) >= (frame_height - 5))

                    if is_full_frame:
                        # É uma detecção de quadro inteiro, o detector falhou.
                        # Pula para a próxima 'face_data' sem desenhar ou salvar.
                        continue
                    
                    # 1. Desenha o retângulo vermelho
                    cv2.rectangle(frame_bgr, (x, y), (x + w, y + h), (0, 0, 255), 2) # (B,G,R), espessura

                    # 2. Prepara o texto e o fundo do label
                    label = emocao
                    # Cria um fundo preenchido para o texto
                    cv2.rectangle(frame_bgr, (x, y), (x + w, y + 25), (0, 0, 255), -1) # -1 = preenchido
                    # Escreve o texto da emoção (branco)
                    cv2.putText(frame_bgr, label, (x + 5, y + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

                except Exception as e:
                    print(f"Erro ao desenhar rosto: {e}") # Ignora rostos mal formatados

            # --- TAREFA 3: Detecção de Atividade ---
            frame_buffer.append(frame_rgb)
            dominant_activity = None

            if len(frame_buffer) == CLIP_LENGTH:
                dominant_activity = analisar_atividade(frame_buffer, model_activity, processor_activity)
                if dominant_activity:
                    all_activities_detected.append(dominant_activity)
                    last_known_activity = dominant_activity # Atualiza a atividade "colada"

                frame_buffer.clear()

            # Escreve a atividade no canto superior esquerdo (em verde)
            cv2.putText(frame_bgr, f"Atividade: {last_known_activity.capitalize()}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # --- TAREFA 4: Armazenamento para Resumo ---
            timeline_summary.append({
                "timestamp": f"{current_time_sec:.2f}s",
                "emocoes": emocoes_no_quadro if emocoes_no_quadro else None,
                "atividade_detectada": dominant_activity # Será 'None' até o buffer encher
            })

            # --- Grava o quadro processado e anotado no vídeo de saída ---
            out_writer.write(frame_bgr)

    # --- Limpeza (Após o loop) ---
    cap.release()
    out_writer.release()
    print("\nProcessamento do vídeo concluído.")
    print(f"Vídeo de saída com anotações salvo em: {OUTPUT_VIDEO_PATH}")


# --- PARTE 3: Geração do Resumo Final ---

print("\n" + "="*30)
print("     RESUMO DA ANÁLISE DO VÍDEO")
print("="*30)

# --- 1. Resumo Estatístico de Emoções ---
print("\n--- Análise de Emoções ---")
if all_emotions_detected:
    total_deteccoes_emocao = len(all_emotions_detected)
    contagem_emocoes = Counter(all_emotions_detected)

    print(f"Total de detecções de emoção: {total_deteccoes_emocao}")
    print("Emoções mais frequentes:")
    for emocao, contagem in contagem_emocoes.most_common(10):
        percentual = (contagem / total_deteccoes_emocao) * 100
        print(f"  - {emocao}: {contagem} vezes ({percentual:.1f}%)")

# --- 2. Resumo Estatístico de Atividades ---
print("\n--- Análise de Atividades ---")
if all_activities_detected:
    total_deteccoes_atividade = len(all_activities_detected)
    contagem_atividades = Counter(all_activities_detected)

    print(f"Total de detecções de atividade: {total_deteccoes_atividade}")
    print("Atividades mais frequentes:")
    for atividade, contagem in contagem_atividades.most_common(10):
        percentual = (contagem / total_deteccoes_atividade) * 100
        print(f"  - {atividade}: {contagem} vezes ({percentual:.1f}%)")

--- Iniciando Célula de Processamento Única ---
Iniciando o processamento do vídeo: video1_original.mp4
Vídeo aberto: 3326 quadros, 30.00 FPS, 1280x720.
Salvando vídeo de saída em 'video_output.mp4' com 30.00 FPS.


Processando e Gravando Vídeo: 100%|██████████| 3326/3326 [03:45<00:00, 14.77it/s]


Processamento do vídeo concluído.
Vídeo de saída com anotações salvo em: video_output.mp4

     RESUMO DA ANÁLISE DO VÍDEO

--- Análise de Emoções ---
Total de detecções de emoção: 4757
Emoções mais frequentes:
  - Sad: 1256 vezes (26.4%)
  - Fear: 1102 vezes (23.2%)
  - Neutral: 990 vezes (20.8%)
  - Happy: 848 vezes (17.8%)
  - Angry: 362 vezes (7.6%)
  - Surprise: 199 vezes (4.2%)

--- Análise de Atividades ---
Total de detecções de atividade: 207
Atividades mais frequentes:
  - trimming or shaving beard: 17 vezes (8.2%)
  - answering questions: 15 vezes (7.2%)
  - crying: 12 vezes (5.8%)
  - massaging person's head: 10 vezes (4.8%)
  - bandaging: 9 vezes (4.3%)
  - shaking hands: 9 vezes (4.3%)
  - yawning: 8 vezes (3.9%)
  - making bed: 8 vezes (3.9%)
  - reading book: 7 vezes (3.4%)
  - bowling: 7 vezes (3.4%)
